In [27]:
%%writefile practice83.cu
// 1. Разделите массив на две части: первая половина обрабатывается на CPU, вторая — на GPU.
// 2. Реализуйте гибридное приложение, которое выполняет обработку массива на CPU и GPU одновременно.
// 3. Замерьте общее время выполнения гибридной обработки.

#include <iostream>         // Подключаем стандартную библиотеку ввода-вывода (cout)
#include <cuda_runtime.h>   // Подключаем библиотеку CUDA для работы с GPU
#include <omp.h>            // Подключаем OpenMP для параллельного выполнения на CPU

using namespace std;        // Чтобы писать cout без std::

__global__ void multiplyByTwoGPU(double* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Вычисляем глобальный индекс потока GPU
    if (idx < N) {                                  // Проверка, чтобы поток не вышел за границы массива
        data[idx] = data[idx] * 2.0f;               // Умножаем элемент массива на 2
    }
}

int main() {                                       // Основная функция
    const int N = 1000000;                         // Общий размер массива
    const int halfN = N / 2;                        // Размер половины массива (для деления CPU/GPU)

    double* h_data = new double[N];                   // Создаем массив на CPU (оперативная память)

    // Инициализация массива значениями 0,1,2,3,... на CPU
    for (int i = 0; i < N; i++) {
        h_data[i] = i * 1.0f;                       // Преобразуем int - float
    }

    double* d_data;                                  // Указатель на память GPU
    cudaMalloc((void**)&d_data, halfN * sizeof(double)); // Выделяем память на GPU только для второй половины массива

    // Копируем вторую половину массива с CPU на GPU
    cudaMemcpy(d_data, h_data + halfN, halfN * sizeof(float), cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;                        // События CUDA для замера времени выполнения
    cudaEventCreate(&start);                        // Создаем событие "старт"
    cudaEventCreate(&stop);                         // Создаем событие "стоп"
    cudaEventRecord(start);                         // Засекаем время начала

    // Параллельная обработка первой половины массива на CPU с OpenMP
    #pragma omp parallel for
    for (int i = 0; i < halfN; i++) {
        h_data[i] = h_data[i] * 2.0f;              // Умножаем каждый элемент первой половины на 2
    }

    // Настройка CUDA ядра для второй половины массива
    int threadsPerBlock = 256;                      // Потоков в блоке
    int blocksPerGrid = (halfN + threadsPerBlock - 1) / threadsPerBlock; // Количество блоков (чтобы покрыть все элементы)
    multiplyByTwoGPU<<<blocksPerGrid, threadsPerBlock>>>(d_data, halfN); // Запуск ядра CUDA

    cudaDeviceSynchronize();                        // Ждем завершения всех потоков GPU

    cudaEventRecord(stop);                          // Засекаем конец
    cudaEventSynchronize(stop);                     // Ждем события stop
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop); // Вычисляем время выполнения (CPU+GPU)

    // Копируем вторую половину обратно с GPU на CPU
    cudaMemcpy(h_data + halfN, d_data, halfN * sizeof(float), cudaMemcpyDeviceToHost);

    // Выводим время выполнения
    cout << "Время выполнения Hybrid CPU + GPU: " << milliseconds / 1000.0f << " с" << endl;

    // Выводим первые 5 элементов массива
    cout << "Первые 5 элементов: ";
    for (int i = 0; i < 5; i++) cout << h_data[i] << " ";
    cout << endl;

    // Выводим последние 5 элементов массива
    cout << "Последние 5 элементов: ";
    for (int i = N-5; i < N; i++) cout << h_data[i] << " ";
    cout << endl;

    cudaFree(d_data);                               // Освобождаем память GPU
    delete[] h_data;                                // Освобождаем память CPU

    return 0;                                       // Конец программы
}



Overwriting practice83.cu


In [38]:
# Компиляция
!nvcc practice83.cu -o practice83 -arch=sm_75 -std=c++11            # -arch=sm_75  - архитектура GPU (Tesla T4 в Colab = sm_75)
                                                                    # -std=c++11 — стандарт C++
# Запуск
!./practice83


Время выполнения Hybrid CPU + GPU: 0.00143715 с
Первые 5 элементов: 0 2 4 6 8 
Последние 5 элементов: 999995 999996 999997 999998 999999 
